In [ ]:
# ============================================================
# INSTALL VOICE TRANSLATOR LIBRARIES
# ============================================================

!pip install -q --upgrade SpeechRecognition
!pip install -q deep-translator
!pip install -q gTTS
!pip install -q pydub

print("✅ Installation completed.")

✅ Installation completed.


In [ ]:
# ============================================================
# TEST LIBRARIES
# ============================================================

import speech_recognition as sr
from deep_translator import GoogleTranslator
from gtts import gTTS
from pydub import AudioSegment

print("✅ SpeechRecognition:", sr.__version__)
print("✅ Deep Translator: OK")
print("✅ gTTS: OK")
print("✅ PyDub: OK")

✅ SpeechRecognition: 3.17.0
✅ Deep Translator: OK
✅ gTTS: OK
✅ PyDub: OK


In [ ]:
# ============================================================
# RECORD AUDIO FROM COMPUTER MICROPHONE
# ============================================================

from IPython.display import Javascript, display
from google.colab import output
import base64

def record_audio(duration=5):

    print("🎤 Requesting microphone access...")
    print(f"⏱️ Recording for {duration} seconds...")

    js_code = f"""
    async function recordAudio() {{
        const stream = await navigator.mediaDevices.getUserMedia({{
            audio: true
        }});

        const recorder = new MediaRecorder(stream);
        const chunks = [];

        recorder.ondataavailable = event => {{
            chunks.push(event.data);
        }};

        const recording = new Promise(resolve => {{
            recorder.onstop = async () => {{
                const blob = new Blob(chunks, {{
                    type: 'audio/webm'
                }});

                const reader = new FileReader();

                reader.onloadend = () => {{
                    resolve(reader.result);
                }};

                reader.readAsDataURL(blob);
            }};
        }});

        recorder.start();

        await new Promise(resolve =>
            setTimeout(resolve, {duration * 1000})
        );

        recorder.stop();

        stream.getTracks().forEach(
            track => track.stop()
        );

        return await recording;
    }}

    recordAudio();
    """

    data = output.eval_js(js_code)

    # Remove data URL prefix
    audio_data = data.split(',')[1]

    # Convert Base64 → bytes
    audio_bytes = base64.b64decode(audio_data)

    # Save recording
    filename = "input_audio.webm"

    with open(filename, "wb") as f:
        f.write(audio_bytes)

    print("✅ Recording completed!")
    print("📁 Saved as:", filename)

    return filename

In [ ]:
# ============================================================
# LANGUAGE CONFIGURATION
# ============================================================

LANGUAGES = {
    "Kannada": {
        "speech_code": "kn-IN",
        "translate_code": "kn"
    },

    "Marathi": {
        "speech_code": "mr-IN",
        "translate_code": "mr"
    },

    "Tamil": {
        "speech_code": "ta-IN",
        "translate_code": "ta"
    },

    "Telugu": {
        "speech_code": "te-IN",
        "translate_code": "te"
    },

    "Bengali": {
        "speech_code": "bn-IN",
        "translate_code": "bn"
    },

    "Gujarati": {
        "speech_code": "gu-IN",
        "translate_code": "gu"
    },

    "Malayalam": {
        "speech_code": "ml-IN",
        "translate_code": "ml"
    },

    "Hindi": {
        "speech_code": "hi-IN",
        "translate_code": "hi"
    },

    "English": {
        "speech_code": "en-IN",
        "translate_code": "en"
    }
}

# Select input and output languages
INPUT_LANGUAGE = "Kannada"
OUTPUT_LANGUAGE = "Hindi"

print("Input Language :", INPUT_LANGUAGE)
print("Output Language:", OUTPUT_LANGUAGE)

In [ ]:
# ============================================================
# RECORD YOUR VOICE
# ============================================================

audio_file = record_audio(5)

🎤 Requesting microphone access...
⏱️ Recording for 5 seconds...
✅ Recording completed!
📁 Saved as: input_audio.webm


In [ ]:
# ============================================================
# CHECK RECORDED AUDIO
# ============================================================

import os

if os.path.exists("input_audio.webm"):

    size = os.path.getsize("input_audio.webm")

    print("✅ Audio file exists")
    print("File size:", size, "bytes")

else:

    print("❌ Audio file was not created")

✅ Audio file exists
File size: 79194 bytes


In [ ]:
# ============================================================
# PLAY RECORDED AUDIO
# ============================================================

from IPython.display import Audio, display

display(
    Audio(
        "input_audio.webm"
    )
)

In [ ]:
# ============================================================
# CONVERT RECORDED AUDIO FROM WEBM TO WAV
# ============================================================

import os
from pydub import AudioSegment

# Check whether the recorded audio exists
if not os.path.exists("input_audio.webm"):

    print("❌ input_audio.webm was not found.")
    print("Please run the microphone recording cell again.")

else:

    print("✅ input_audio.webm found.")

    # Load the recorded WebM audio
    audio = AudioSegment.from_file(
        "input_audio.webm",
        format="webm"
    )

    # Export as WAV
    audio.export(
        "input_audio.wav",
        format="wav"
    )

    print("✅ Audio converted successfully.")

    # Verify WAV file
    if os.path.exists("input_audio.wav"):

        file_size = os.path.getsize("input_audio.wav")

        print("✅ input_audio.wav created.")
        print("File size:", file_size, "bytes")

    else:

        print("❌ WAV file was not created.")

✅ input_audio.webm found.
✅ Audio converted successfully.
✅ input_audio.wav created.
File size: 944684 bytes


In [ ]:
# ============================================================
# CONVERT VOICE TO TEXT
# ============================================================

import speech_recognition as sr

recognizer = sr.Recognizer()

print("🔄 Loading audio...")

with sr.AudioFile("input_audio.wav") as source:
    audio_data = recognizer.record(source)

print("🔄 Recognizing Kannada speech...")
print("Please wait...")

try:

    text = recognizer.recognize_google(
        audio_data,
        language="kn-IN"
    )

    print("\n======================================")
    print("🎤 RECOGNIZED KANNADA TEXT")
    print("======================================")

    print(text)

except sr.UnknownValueError:

    print("\n❌ Could not understand the speech.")
    print("Please try recording again.")

except sr.RequestError as e:

    print("\n❌ Speech recognition service error:")
    print(e)

🔄 Loading audio...
🔄 Recognizing Kannada speech...
Please wait...

🎤 RECOGNIZED KANNADA TEXT
ನಿನ್ನ ಹೆಸರೇನು


In [ ]:
# ============================================================
# TRANSLATE KANNADA TO HINDI
# ============================================================

from deep_translator import GoogleTranslator

print("🔄 Translating Kannada → Hindi...")
print()

try:

    translated_text = GoogleTranslator(
        source="kn",
        target="hi"
    ).translate(text)

    print("======================================")
    print("🇮🇳 HINDI TRANSLATION")
    print("======================================")

    print(translated_text)

except Exception as e:

    print("❌ Translation error:")
    print(e)

🔄 Translating Kannada → Hindi...

🇮🇳 HINDI TRANSLATION
आपका क्या नाम है


In [ ]:
# ============================================================
# CONVERT HINDI TEXT TO HINDI VOICE
# ============================================================

from gtts import gTTS

output_file = "hindi_output.mp3"

print("🔊 Generating Hindi voice...")

try:

    tts = gTTS(
        text=translated_text,
        lang="hi"
    )

    tts.save(output_file)

    print("✅ Hindi voice generated successfully!")
    print("📁 File:", output_file)

except Exception as e:

    print("❌ Text-to-speech error:")
    print(e)

🔊 Generating Hindi voice...
✅ Hindi voice generated successfully!
📁 File: hindi_output.mp3


In [ ]:
# ============================================================
# PLAY HINDI VOICE
# ============================================================

from IPython.display import Audio, display

print("🔊 Playing Hindi translation...")

display(
    Audio(
        "hindi_output.mp3",
        autoplay=True
    )
)

🔊 Playing Hindi translation...
